In [1]:
import os
import pickle
import torch

DATA_DIR = "processed_data"

def load_pickle(filename):
    with open(os.path.join(DATA_DIR, filename), "rb") as f:
        return pickle.load(f)

train_encodings = load_pickle("train_encodings.pkl")
val_encodings   = load_pickle("val_encodings.pkl")
test_encodings  = load_pickle("test_encodings.pkl")

train_mapping = load_pickle("train_mapping.pkl")
val_mapping   = load_pickle("val_mapping.pkl")
test_mapping  = load_pickle("test_mapping.pkl")

print("Train chunks:", len(train_mapping))
print("Validation chunks:", len(val_mapping))
print("Test chunks:", len(test_mapping))

Train chunks: 334755
Validation chunks: 41833
Test chunks: 41744


In [2]:
from torch.utils.data import Dataset

class PromptChunkDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(
                self.encodings["input_ids"][idx],
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                self.encodings["attention_mask"][idx],
                dtype=torch.long
            ),
            "labels": torch.tensor(
                self.encodings["labels"][idx],
                dtype=torch.long
            )
        }

train_dataset = PromptChunkDataset(train_encodings)
val_dataset   = PromptChunkDataset(val_encodings)
test_dataset  = PromptChunkDataset(test_encodings)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 334755
Validation dataset: 41833
Test dataset: 41744


In [3]:
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

print(model.config)

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.15.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [4]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./bert_tiny_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,

    num_train_epochs=3,

    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=500,

    save_total_limit=2,

    report_to="none",

    fp16=torch.cuda.is_available(),

    seed=42
)

In [5]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [6]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.119001,0.109678,0.959051,0.969535,0.953583,0.961493
2,0.101905,0.095524,0.965745,0.970971,0.964953,0.967953
3,0.083439,0.095093,0.967394,0.975098,0.963794,0.969413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=31386, training_loss=0.1257895984575518, metrics={'train_runtime': 534.676, 'train_samples_per_second': 1878.268, 'train_steps_per_second': 58.701, 'total_flos': 318976416806400.0, 'train_loss': 0.1257895984575518, 'epoch': 3.0})

In [7]:
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# --------------------------------------------------
# Generate predictions
# --------------------------------------------------

print("Generating validation predictions...")

val_output = trainer.predict(val_dataset)

print("Generating test predictions...")

test_output = trainer.predict(test_dataset)

# Logits
val_logits = val_output.predictions
test_logits = test_output.predictions

print("Validation logits shape:", val_logits.shape)
print("Test logits shape:", test_logits.shape)

Generating validation predictions...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Generating test predictions...


C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(
C:\Users\Jivesh Gawde\AppData\Local\Temp\ipykernel_27872\3888382844.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(


Validation logits shape: (41833, 2)
Test logits shape: (41744, 2)


In [8]:
def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


val_probs = softmax(val_logits)[:, 1]
test_probs = softmax(test_logits)[:, 1]

print("First 10 validation probabilities:")
print(val_probs[:10])

print("\nFirst 10 test probabilities:")
print(test_probs[:10])

First 10 validation probabilities:
[8.3935994e-04 6.8383530e-04 1.3617034e-03 8.0092567e-01 8.0095365e-04
 9.9935383e-01 9.9932146e-01 8.1038737e-04 9.9845421e-01 9.9890769e-01]

First 10 test probabilities:
[9.9882132e-01 2.0553529e-01 9.9928480e-01 9.9760324e-01 9.9938214e-01
 1.1013573e-01 7.2790676e-04 7.2083797e-04 4.4554460e-01 9.9252492e-01]


In [9]:
def aggregate_predictions(probs, mapping, method="max", threshold=0.5):
    """
    Convert chunk-level probabilities into prompt-level predictions.

    probs:
        Chunk-level malicious probabilities.

    mapping:
        overflow_to_sample_mapping. Each chunk maps to
        its original prompt index.

    method:
        "max", "mean", or "majority"

    threshold:
        Probability threshold for binary classification.
    """

    mapping = mapping.cpu().numpy() if torch.is_tensor(mapping) else np.asarray(mapping)

    # Number of original prompts
    n_prompts = int(mapping.max()) + 1

    prompt_probs = np.zeros(n_prompts)

    for prompt_idx in range(n_prompts):

        chunk_probs = probs[mapping == prompt_idx]

        if method == "max":
            prompt_probs[prompt_idx] = np.max(chunk_probs)

        elif method == "mean":
            prompt_probs[prompt_idx] = np.mean(chunk_probs)

        elif method == "majority":
            chunk_predictions = (chunk_probs >= threshold).astype(int)
            prompt_probs[prompt_idx] = np.mean(chunk_predictions)

        else:
            raise ValueError(
                "method must be 'max', 'mean', or 'majority'"
            )

    if method == "majority":
        prompt_predictions = (prompt_probs >= 0.5).astype(int)

    else:
        prompt_predictions = (prompt_probs >= threshold).astype(int)

    return prompt_probs, prompt_predictions

In [10]:
def calculate_metrics(y_true, y_pred):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "FNR": fnr,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    }

In [11]:
test_df = pd.read_csv("test.csv")
val_df = pd.read_csv("validation.csv")

In [12]:
y_val = val_df["label"].astype(int).to_numpy()

validation_results = {}

for method in ["max", "mean", "majority"]:

    _, val_predictions = aggregate_predictions(
        val_probs,
        val_mapping,
        method=method
    )

    metrics = calculate_metrics(
        y_val,
        val_predictions
    )

    validation_results[method] = metrics


validation_table = pd.DataFrame(
    validation_results
).T

print("\nPROMPT-LEVEL VALIDATION RESULTS")
print("=" * 70)

print(
    validation_table[
        [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "FPR",
            "FNR"
        ]
    ].round(4)
)


PROMPT-LEVEL VALIDATION RESULTS
          Accuracy  Precision  Recall      F1     FPR     FNR
max         0.9761     0.9754  0.9793  0.9773  0.0274  0.0207
mean        0.9775     0.9803  0.9769  0.9786  0.0218  0.0231
majority    0.9764     0.9764  0.9787  0.9776  0.0262  0.0213


In [13]:
best_method = validation_table["F1"].idxmax()

print("\nBest aggregation method:")
print(best_method)

print(
    "Validation F1:",
    validation_table.loc[best_method, "F1"]
)


Best aggregation method:
mean
Validation F1: 0.9785993750492346


In [14]:
y_test = test_df["label"].astype(int).to_numpy()

test_prompt_probs, test_predictions = aggregate_predictions(
    test_probs,
    test_mapping,
    method=best_method
)

test_metrics = calculate_metrics(
    y_test,
    test_predictions
)

print("\n" + "=" * 70)
print("BERT-TINY FINAL PROMPT-LEVEL TEST RESULTS")
print("=" * 70)

for metric, value in test_metrics.items():

    if metric in ["TN", "FP", "FN", "TP"]:
        print(f"{metric:10s}: {value}")

    else:
        print(f"{metric:10s}: {value:.4f}")


BERT-TINY FINAL PROMPT-LEVEL TEST RESULTS
Accuracy  : 0.9763
Precision : 0.9789
Recall    : 0.9759
F1        : 0.9774
FPR       : 0.0234
FNR       : 0.0241
TN        : 16796
FP        : 402
FN        : 459
TP        : 18615


In [15]:
print("\nTEST SANITY CHECK")
print("-" * 50)

print("Original test prompts :", len(test_df))
print("Test chunks            :", len(test_mapping))
print("Prompt probabilities   :", len(test_prompt_probs))
print("Prompt predictions     :", len(test_predictions))
print("Ground-truth labels    :", len(y_test))


TEST SANITY CHECK
--------------------------------------------------
Original test prompts : 36272
Test chunks            : 41744
Prompt probabilities   : 36272
Prompt predictions     : 36272
Ground-truth labels    : 36272
